In [ ]:
!pip install pandas scikit-learn kagglehub

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

# Generate car details
year = np.random.randint(2005, 2025, n)
km_driven = np.random.randint(5000, 200000, n)

fuel = np.random.choice(
    ["Petrol", "Diesel", "CNG"],
    n,
    p=[0.55, 0.35, 0.10]
)

seller_type = np.random.choice(
    ["Individual", "Dealer"],
    n
)

transmission = np.random.choice(
    ["Manual", "Automatic"],
    n,
    p=[0.7, 0.3]
)

owner = np.random.choice(
    ["First Owner", "Second Owner", "Third Owner"],
    n,
    p=[0.65, 0.25, 0.10]
)

present_price = np.random.uniform(3, 25, n)

# Calculate vehicle age
age = 2025 - year

# Generate selling prices (in lakhs)
selling_price = (
    present_price * (0.88 ** age)
    - km_driven * 0.000008
    + np.where(fuel == "Diesel", 0.5, 0)
    + np.where(transmission == "Automatic", 0.7, 0)
    + np.where(owner == "First Owner", 0.3, -0.3)
    + np.random.normal(0, 0.7, n)
)

# Ensure positive selling prices
selling_price = np.maximum(selling_price, 0.5)

# Add a few expensive-car outliers for Q10
outlier_indices = np.random.choice(n, 15, replace=False)
selling_price[outlier_indices] *= 2.5

# Create DataFrame
df = pd.DataFrame({
    "Year": year,
    "Present_Price": present_price.round(2),
    "Kms_Driven": km_driven,
    "Fuel_Type": fuel,
    "Seller_Type": seller_type,
    "Transmission": transmission,
    "Owner": owner,
    "Selling_Price": selling_price.round(2)
})

# Save dataset
df.to_csv("used_cars.csv", index=False)

print("Dataset created successfully!")
print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset created successfully!
   Year  Present_Price  Kms_Driven Fuel_Type Seller_Type Transmission  \
0  2011          10.83       13070    Petrol  Individual       Manual   
1  2024          17.97      131161    Petrol  Individual       Manual   
2  2019           9.03       13666    Petrol  Individual       Manual   
3  2015          21.21       62761    Petrol  Individual       Manual   
4  2012          19.28       41368    Diesel  Individual       Manual   

          Owner  Selling_Price  
0   First Owner           2.34  
1   First Owner          16.09  
2   First Owner           4.59  
3  Second Owner           4.47  
4   First Owner           4.57  

Shape: (1000, 8)

Columns: ['Year', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'Selling_Price']


In [ ]:
df = pd.read_csv('used_cars.csv')
df.head()

,Year,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Selling_Price
0,2011,10.83,13070,Petrol,Individual,Manual,First Owner,2.34
1,2024,17.97,131161,Petrol,Individual,Manual,First Owner,16.09
2,2019,9.03,13666,Petrol,Individual,Manual,First Owner,4.59
3,2015,21.21,62761,Petrol,Individual,Manual,Second Owner,4.47
4,2012,19.28,41368,Diesel,Individual,Manual,First Owner,4.57


In [ ]:
print("Missing values:", df.isnull().sum().sum(), "| Duplicates:", df.duplicated().sum())

Missing values: 0 | Duplicates: 0


In [ ]:
df = df.dropna().drop_duplicates()
df = df.drop(columns=["Present_Price", "Seller_Type", "Fuel_Type"])

df.head()

,Year,Kms_Driven,Transmission,Owner,Selling_Price
0,2011,13070,Manual,First Owner,2.34
1,2024,131161,Manual,First Owner,16.09
2,2019,13666,Manual,First Owner,4.59
3,2015,62761,Manual,Second Owner,4.47
4,2012,41368,Manual,First Owner,4.57


In [ ]:
df =  pd.get_dummies(df, columns=["Transmission", "Owner"], drop_first=True, dtype=int)
df.head()

,Year,Kms_Driven,Selling_Price,Transmission_Manual,Owner_Second Owner,Owner_Third Owner
0,2011,13070,2.34,1,0,0
1,2024,131161,16.09,1,0,0
2,2019,13666,4.59,1,0,0
3,2015,62761,4.47,1,1,0
4,2012,41368,4.57,1,0,0


In [ ]:
from sklearn.model_selection import train_test_split
x = df.drop(columns=["Selling_Price"])
y = df["Selling_Price"]

x_train, x_test, y_train, y_test = train_test_split (x, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

models = {"Linear Regression": LinearRegression(), "Decision Tree": DecisionTreeRegressor(), "Random Forest": RandomForestRegressor()}

for name, model in models.items():
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  rmse = np.sqrt(mean_squared_error(y_test, pred))
  r2 = r2_score(y_test, pred)
  print(f"{name}: RMSE = {rmse:.2f}, R2 = {r2:.2f}")

Linear Regression: RMSE = 2.71, R2 = 0.55
Decision Tree: RMSE = 3.79, R2 = 0.13
Random Forest: RMSE = 2.86, R2 = 0.50
